#### 0.环境与路径

In [ ]:
import os, time, csv, math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("使用设备：", DEVICE)

In [ ]:
WHO = "B"  # 我的结果文件
OUT_DIR = "./outputs"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/diffusion_project"
except Exception:
    pass
os.makedirs(OUT_DIR, exist_ok=True)
RESULT_CSV = os.path.join(OUT_DIR, f"results_{WHO}.csv")

#### 1.参数
以后续冻结参数为准

In [ ]:
CFG = dict(
    K=8,            # 模式数（圆周上等距的高斯分量数）
    sigma=0.3,      # 每个分量的标准差（固定）
    n_train=1000,   # 训练集大小
    T=100,          # 扩散步数
    iters=10000,     # 训练迭代次数
    batch=256,      # 批大小
    lr=1e-3,        # 学习率
    n_sample=2000,  # 每次采样生成的点数
    n_traj=10,      # 轨迹快照数（含起点纯噪声和终点最终样本）
    tdim=32,        # 时间嵌入维度
)
DISTANCES  = [1.0, 1.5, 2.0, 3.0, 4.5]   # frozen
CAPACITIES = [(1, 8), (2, 16), (3, 64), (4, 256), (5, 512)]   # frozen (depth, width)
SEEDS = [0, 1, 2, 3, 4]
##### 三份样本集的种子约定 #####
# 训练集种子 = seed；密度校准集 = 10000 + seed；目标参考集 = 20000 + seed

#### 2.数据生成器

In [ ]:
##### 圆周等距均值的二维高斯混合 #####
def make_gmm(n, K, radius, sigma, seed):
    """返回 n 个二维点、每个点所属分量编号、K 个均值。radius 越大，模式离得越远。"""
    rng = np.random.default_rng(seed)
    ang = 2 * np.pi * np.arange(K) / K
    means = radius * np.stack([np.cos(ang), np.sin(ang)], axis=1)
    comp = rng.integers(0, K, size=n)
    x = means[comp] + sigma * rng.standard_normal((n, 2))
    return x.astype(np.float32), comp, means

#### 3.模型与DDPM

In [ ]:
class Denoiser(nn.Module):
    """MLP 去噪网络：输入 (带噪点, 时间步)，输出预测的噪声。depth=隐藏层数，width=每层宽度。"""
    def __init__(self, depth, width, tdim=32):
        super().__init__()
        self.tdim = tdim
        layers, d_in = [], 2 + tdim
        for i in range(depth):
            layers += [nn.Linear(d_in if i == 0 else width, width), nn.SiLU()]
        layers.append(nn.Linear(width, 2))
        self.net = nn.Sequential(*layers)

    def temb(self, t):
        half = self.tdim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        a = t.float()[:, None] * freqs[None]
        return torch.cat([a.sin(), a.cos()], dim=1)

    def forward(self, x, t):
        return self.net(torch.cat([x, self.temb(t)], dim=1))

In [ ]:
class DDPM:
    """线性噪声日程；beta 按步数缩放，保证无论 T 多大，最后一步都接近纯噪声。"""
    def __init__(self, T, device):
        self.T = T
        s = 1000.0 / T
        self.betas = torch.linspace(1e-4 * s, 0.02 * s, T, device=device)
        self.alphas = 1 - self.betas
        self.abar = torch.cumprod(self.alphas, dim=0)

    def loss(self, model, x0):
        t = torch.randint(0, self.T, (x0.shape[0],), device=x0.device)
        eps = torch.randn_like(x0)
        ab = self.abar[t][:, None]
        xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
        return ((model(xt, t) - eps) ** 2).mean()


def build_model(depth, width, cfg):
    model = Denoiser(depth, width, cfg["tdim"]).to(DEVICE)
    model.ddpm = DDPM(cfg["T"], DEVICE)   # 把扩散参数挂在模型上，采样时直接用
    return model


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

#### 4.训练、采样、单次运行、批量调度

In [ ]:
def train(depth, width, radius, seed, cfg, ckpt_path=None):
    """训练一个模型。每 1000 步存断点，断线后重跑会从断点继续。"""
    set_seed(seed)
    x_np, _, _ = make_gmm(cfg["n_train"], cfg["K"], radius, cfg["sigma"], seed)
    x = torch.tensor(x_np, device=DEVICE)
    model = build_model(depth, width, cfg)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    start, losses = 0, []
    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        start, losses = ck["it"], ck["losses"]
        print(f"  从断点继续：第 {start} 步")
    model.train()
    bs = min(cfg["batch"], len(x))
    for it in range(start, cfg["iters"]):
        idx = torch.randint(0, len(x), (bs,), device=DEVICE)
        loss = model.ddpm.loss(model, x[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if ckpt_path and (it + 1) % 1000 == 0:
            torch.save(dict(model=model.state_dict(), opt=opt.state_dict(),
                            it=it + 1, losses=losses), ckpt_path)
    return model, losses, x_np


@torch.no_grad()
def sample(model, n, return_trajectory=True, seed=0, n_traj=10):
    """
    和 A 约定的采样接口：
      final = sample(model, n, return_trajectory=False)            -> (n, 2)
      final, traj = sample(model, n, return_trajectory=True)       -> traj 形状 (n_traj, n, 2)
    traj[0] 是纯噪声，traj[-1] 就是最终样本，中间是反向扩散过程中等间隔的快照。
    """
    model.eval()
    d = model.ddpm
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    x = torch.randn(n, 2, device=DEVICE, generator=g)
    keep = set(np.linspace(d.T - 1, 0, n_traj - 1).round().astype(int).tolist())
    traj = [x.cpu().numpy()]
    for t in range(d.T - 1, -1, -1):
        tt = torch.full((n,), t, device=DEVICE, dtype=torch.long)
        eps = model(x, tt)
        a, ab, b = d.alphas[t], d.abar[t], d.betas[t]
        mean = (x - b / (1 - ab).sqrt() * eps) / a.sqrt()
        if t > 0:
            x = mean + b.sqrt() * torch.randn(x.shape, device=DEVICE, generator=g)
        else:
            x = mean
        if t in keep:
            traj.append(x.cpu().numpy())
    final = x.cpu().numpy()
    if return_trajectory:
        return final, np.stack(traj)
    return final

In [ ]:
##### 结果文件：一行一次运行。后面三列留空，由 A 的评估代码回填。
FIELDS = ["tag", "dist_idx", "cap_idx", "radius", "depth", "width", "params", "seed",
          "n_train", "T", "K", "final_loss", "seconds", "npz_path",
          "mode_coverage", "mmd", "nn_ratio"]

def append_row(row):
    new = not os.path.exists(RESULT_CSV)
    with open(RESULT_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow({k: row.get(k, "") for k in FIELDS})


def run_one(di, ci, seed, cfg=CFG, tag="main"):
    """跑一个配置（距离档 di、容量档 ci、种子 seed）。已完成的自动跳过。"""
    radius = DISTANCES[di]
    depth, width = CAPACITIES[ci]
    name = f"{tag}_d{di}_c{ci}_s{seed}"
    folder = os.path.join(OUT_DIR, tag)
    os.makedirs(folder, exist_ok=True)
    npz = os.path.join(folder, name + ".npz")
    ckpt = os.path.join(folder, name + ".ckpt")
    if os.path.exists(npz):
        print("跳过（已完成）：", name)
        return
    t0 = time.time()
    model, losses, x_train = train(depth, width, radius, seed, cfg, ckpt)
    final, traj = sample(model, cfg["n_sample"], True, seed=seed, n_traj=cfg["n_traj"])
    np.savez_compressed(npz, final=final, traj=traj, train=x_train,
                        losses=np.array(losses), means_radius=radius)
    torch.save(model.state_dict(), os.path.join(folder, name + ".pt"))
    sec = time.time() - t0
    append_row(dict(tag=tag, dist_idx=di, cap_idx=ci, radius=radius, depth=depth, width=width,
                    params=n_params(model), seed=seed, n_train=cfg["n_train"], T=cfg["T"],
                    K=cfg["K"], final_loss=round(float(np.mean(losses[-200:])), 5),
                    seconds=round(sec, 1), npz_path=npz))
    if os.path.exists(ckpt):
        os.remove(ckpt)
    print(f"完成 {name}：参数量 {n_params(model)}，用时 {sec:.1f}s")


def run_grid(dist_idx_list, cap_idx_list=range(5), seeds=SEEDS, cfg=CFG, tag="main"):
    """批量跑网格。种子放最外层：先把所有格子各跑1个种子，万一时间不够，每格至少有数据。"""
    for s in seeds:
        for di in dist_idx_list:
            for ci in cap_idx_list:
                run_one(di, ci, s, cfg, tag)


def run_ablation(key, values, di=2, ci=2, seeds=(0, 1)):
    """消融：固定一个中间配置，只改 cfg 里的一项（如 n_train、T、K）。"""
    for v in values:
        cfg = dict(CFG); cfg[key] = v
        for s in seeds:
            run_one(di, ci, s, cfg, tag=f"abl_{key}{v}")

#### 5.自检

In [ ]:
def selfcheck(di=3, ci=2, seed=0):
    t0 = time.time()
    model, losses, x_train = train(CAPACITIES[ci][0], CAPACITIES[ci][1], DISTANCES[di], seed, CFG)
    print(f"训练用时 {time.time()-t0:.1f}s，参数量 {n_params(model)}，最后200步平均损失 {np.mean(losses[-200:]):.4f}")

    final, traj = sample(model, 2000, True, seed=seed)
    final2, _ = sample(model, 2000, True, seed=seed)
    print("同种子采样可复现：", np.allclose(final, final2))
    print("轨迹末尾等于最终样本：", np.allclose(traj[-1], final))
    print("轨迹形状：", traj.shape, "（应为 (10, 2000, 2)）")
    print("起点标准差（应接近1）：", traj[0].std().round(3))

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    ax[0].plot(losses); ax[0].set_title("训练损失"); ax[0].set_yscale("log")
    ax[1].scatter(x_train[:, 0], x_train[:, 1], s=3, label="训练集")
    ax[1].scatter(final[:, 0], final[:, 1], s=3, alpha=.5, label="生成样本")
    ax[1].legend(); ax[1].set_aspect("equal"); ax[1].set_title("训练集 vs 生成样本")
    for k in range(0, traj.shape[0], 3):
        ax[2].scatter(traj[k][:300, 0], traj[k][:300, 1], s=3, label=f"快照{k}")
    ax[2].legend(); ax[2].set_aspect("equal"); ax[2].set_title("反向扩散轨迹")
    plt.tight_layout(); plt.show()
    np.savez(os.path.join(OUT_DIR, "selfcheck_samples.npz"), final=final, traj=traj, train=x_train)
    print("样本已存：", os.path.join(OUT_DIR, "selfcheck_samples.npz"))

selfcheck()

#### 6.pilot

pilot 的目的：找到"模式崩塌的距离区间"和"泛化饱和的容量点"，用来决定正式的五档。

###### pilot 1

In [ ]:
##### PILOT：4 个距离 × 4 个容量，1 个种子 #####
DISTANCES  = [0.5, 1.0, 2.0, 4.0]
CAPACITIES = [(1, 8), (2, 16), (3, 64), (4, 256)]

for di in range(4):
    for ci in range(4):
        run_one(di, ci, seed=0, tag="pilot")

# 画 4×4 散点图总览
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for di in range(4):
    for ci in range(4):
        d = np.load(f"{OUT_DIR}/pilot/pilot_d{di}_c{ci}_s0.npz")
        ax = axes[di, ci]
        ax.scatter(d["train"][:, 0], d["train"][:, 1], s=2, c="tab:blue")
        ax.scatter(d["final"][:, 0], d["final"][:, 1], s=2, c="tab:orange", alpha=.5)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"距离{DISTANCES[di]} 深{CAPACITIES[ci][0]}宽{CAPACITIES[ci][1]}", fontsize=8)
plt.tight_layout(); plt.show()

##### pilot 2

加容量

In [ ]:
DISTANCES  = [1.0, 1.5, 2.0, 3.0, 4.5]
CAPACITIES = [(1, 8), (2, 16), (3, 64), (4, 256), (5, 512)]

for di in range(5):
    for ci in range(5):
        run_one(di, ci, seed=0, tag="pilot2")

fig, axes = plt.subplots(5, 5, figsize=(16, 16))
for di in range(5):
    for ci in range(5):
        d = np.load(f"{OUT_DIR}/pilot2/pilot2_d{di}_c{ci}_s0.npz")
        ax = axes[di, ci]
        ax.scatter(d["train"][:, 0], d["train"][:, 1], s=2, c="tab:blue")
        ax.scatter(d["final"][:, 0], d["final"][:, 1], s=2, c="tab:orange", alpha=.5)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"r={DISTANCES[di]} d{CAPACITIES[ci][0]}w{CAPACITIES[ci][1]}", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
##### 对比判断训练迭代次数是否够（同一配置迭代10000 步）
cfg_long = dict(CFG, iters=10000)
for ci in [3, 4]:                       # 最大的两档容量
    run_one(2, ci, seed=0, cfg=cfg_long, tag="pilot_iters10k")   # 距离 2.0

fig, axes = plt.subplots(2, 2, figsize=(9, 9))
for r, ci in enumerate([3, 4]):
    for c, (tag, nm) in enumerate([("pilot2", "3000步"), ("pilot_iters10k", "10000步")]):
        d = np.load(f"{OUT_DIR}/{tag}/{tag}_d2_c{ci}_s0.npz")
        ax = axes[r, c]
        ax.scatter(d["train"][:, 0], d["train"][:, 1], s=2, c="tab:blue")
        ax.scatter(d["final"][:, 0], d["final"][:, 1], s=2, c="tab:orange", alpha=.5)
        ax.set_aspect("equal"); ax.set_title(f"cap{ci} {nm}", fontsize=9)
plt.tight_layout(); plt.show()

#### 7.冻结参数

In [ ]:
DISTANCES  = [1.0, 1.5, 2.0, 3.0, 4.5]
CAPACITIES = [(1, 8), (2, 16), (3, 64), (4, 256), (5, 512)]
SEEDS = [0, 1, 2, 3, 4]

CFG["iters"] = 10000

print("iters:", CFG["iters"], "| 默认配置里的 iters:", run_one.__defaults__[0]["iters"])
print("距离:", DISTANCES)
print("容量:", CAPACITIES)

#### 8.批量跑主实验

In [ ]:
##### 距离第 4、5 档（下标 3、4）#####
run_grid(dist_idx_list=[3, 4], seeds=SEEDS, cfg=CFG)

In [ ]:
##### 核对 #####
import glob, pandas as pd

files = glob.glob(f"{OUT_DIR}/main/main_d[34]_c*_s*.npz")
print("main 的 npz 数量：", len(files))                  # 应为 50

df = pd.read_csv(RESULT_CSV)
m = df[df.tag == "main"]
print("csv 里 main 的行数：", len(m))                     # 应为 50
print("重复行数：", m.duplicated(["dist_idx", "cap_idx", "seed"]).sum())   # 应为 0
print(m.groupby(["dist_idx", "cap_idx"]).size().unstack())               # 每格应为 5

In [ ]:
##### 距离第1、2、3档（下标0、1、2） #####
print("iters:", CFG["iters"], "| n_train:", CFG["n_train"])   # 应为 10000 | 1000
print("距离:", DISTANCES)                                      # [1.0, 1.5, 2.0, 3.0, 4.5]
print("容量:", CAPACITIES)                                     # 5 档，最后是 (5, 512)
print("默认配置 iters:", run_one.__defaults__[0]["iters"])     # 10000

run_grid(dist_idx_list=[0, 1, 2], seeds=SEEDS, cfg=CFG, tag="main")

In [ ]:
##### 核对 #####
files = glob.glob(f"{OUT_DIR}/main/main_d*_c*_s*.npz")
print("npz 数量：", len(files))                                # 应为 125
df = pd.read_csv(RESULT_CSV); m = df[df.tag == "main"]
print("csv 行数：", len(m), "| 重复：", m.duplicated(["dist_idx","cap_idx","seed"]).sum())
print(m.groupby(["dist_idx","cap_idx"]).size().unstack())     # 全是 5

##### 可视化

In [ ]:
import matplotlib.pyplot as plt, numpy as np

# 图 1：样本散点图（3 距离 × 3 容量）
DI = [0, 2, 4]; CI = [0, 2, 4]
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for r, di in enumerate(DI):
    for c, ci in enumerate(CI):
        d = np.load(f"{OUT_DIR}/main/main_d{di}_c{ci}_s0.npz")
        ax = axes[r, c]
        ax.scatter(d["train"][:, 0], d["train"][:, 1], s=2, c="tab:blue")
        ax.scatter(d["final"][:, 0], d["final"][:, 1], s=2, c="tab:orange", alpha=.4)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"r={DISTANCES[di]}, d{CAPACITIES[ci][0]}w{CAPACITIES[ci][1]}", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_scatter.png", dpi=200); plt.show()

# 图 2：训练曲线（距离 2.0，种子 0，五档容量）
def smooth(x, k=200):
    return np.convolve(x, np.ones(k) / k, mode="valid")
plt.figure(figsize=(6, 4))
for ci in range(5):
    d = np.load(f"{OUT_DIR}/main/main_d2_c{ci}_s0.npz")
    plt.plot(smooth(d["losses"]), label=f"d{CAPACITIES[ci][0]}w{CAPACITIES[ci][1]}")
plt.yscale("log"); plt.xlabel("训练步数"); plt.ylabel("损失（200 步滑动平均）")
plt.legend(); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_loss.png", dpi=200); plt.show()

#### 9.记忆测试

In [ ]:
CAPACITIES_BACKUP = list(CAPACITIES)
CAPACITIES.append((4, 1024))          # 临时加第 6 档（下标 5，约 320 万参数）

for n in [16, 40, 100, 300]:
    cfg = dict(CFG, n_train=n, iters=30000)
    for ci in [4, 5]:                 # 100 万 和 320 万参数
        run_one(3, ci, seed=0, cfg=cfg, tag=f"memtest_n{n}")   # 距离 3.0

CAPACITIES[:] = CAPACITIES_BACKUP     # 恢复，不影响后面的主实验

In [ ]:
print(len(CAPACITIES), CAPACITIES)   # 应该是 5 个，最后一个是 (5, 512)

In [ ]:
from scipy.spatial import cKDTree

fig, axes = plt.subplots(4, 2, figsize=(9, 18))
print("n_train | 容量 | 生成点中位近邻距离 | 真实新样本中位近邻距离 | 比值")
for r, n in enumerate([16, 40, 100, 300]):
    for c, ci in enumerate([4, 5]):
        d = np.load(f"{OUT_DIR}/memtest_n{n}/memtest_n{n}_d3_c{ci}_s0.npz")
        tr, gen = d["train"], d["final"]
        fresh, _, _ = make_gmm(2000, 8, 3.0, 0.3, 20000)       # 距离 3.0 的新样本
        tree = cKDTree(tr)
        dg = np.median(tree.query(gen)[0])
        df = np.median(tree.query(fresh)[0])
        print(f"{n:>5} | cap{ci} | {dg:.3f} | {df:.3f} | {dg/df:.2f}")
        ax = axes[r, c]
        ax.scatter(gen[:, 0], gen[:, 1], s=2, c="tab:orange", alpha=.4)
        ax.scatter(tr[:, 0], tr[:, 1], s=14, c="tab:blue")
        ax.set_aspect("equal"); ax.set_title(f"n={n}, cap{ci}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
print(len(CAPACITIES), CAPACITIES)   # 应为 5 个，最后是 (5, 512)
# 如果是 6 个：CAPACITIES.pop()

#### 10.批量跑mainN40

In [ ]:
##### mainN40（距离下标 3、4）#####
cfg_mem = dict(CFG, n_train=40, iters=30000)
print("检查：", cfg_mem["n_train"], cfg_mem["iters"])      # 应为 40 30000
print(len(CAPACITIES), CAPACITIES)                        # 5 档，最后是 (5, 512)

run_grid(dist_idx_list=[3, 4], seeds=SEEDS, cfg=cfg_mem, tag="mainN40")

In [ ]:
##### 核对 #####
files = glob.glob(f"{OUT_DIR}/mainN40/mainN40_d[34]_c*_s*.npz")
print("npz 数量：", len(files))                       # 应为 50
df = pd.read_csv(RESULT_CSV); m = df[df.tag == "mainN40"]
print("csv 行数：", len(m), "| 重复：", m.duplicated(["dist_idx","cap_idx","seed"]).sum())
print(m.groupby(["dist_idx","cap_idx"]).size().unstack())    # 应为 2 行 5 列，全是 5

#### 11.宽度扫描

In [ ]:
def run_custom(radius, depth, width, seed, cfg, tag):
    """任意（深度，宽度）的运行，参数与 main 相同。"""
    name = f"{tag}_r{radius}_d{depth}_w{width}_s{seed}"
    folder = os.path.join(OUT_DIR, tag); os.makedirs(folder, exist_ok=True)
    npz = os.path.join(folder, name + ".npz")
    ckpt = os.path.join(folder, name + ".ckpt")
    if os.path.exists(npz):
        print("跳过（已完成）：", name); return
    t0 = time.time()
    model, losses, x_train = train(depth, width, radius, seed, cfg, ckpt)
    final, traj = sample(model, cfg["n_sample"], True, seed=seed, n_traj=cfg["n_traj"])
    np.savez_compressed(npz, final=final, traj=traj, train=x_train,
                        losses=np.array(losses), means_radius=radius)
    append_row(dict(tag=tag, dist_idx=DISTANCES.index(radius), cap_idx=-1, radius=radius,
                    depth=depth, width=width, params=n_params(model), seed=seed,
                    n_train=cfg["n_train"], T=cfg["T"], K=cfg["K"],
                    final_loss=round(float(np.mean(losses[-200:])), 5),
                    seconds=round(time.time() - t0, 1), npz_path=npz))
    if os.path.exists(ckpt): os.remove(ckpt)
    print(f"完成 {name}：参数量 {n_params(model)}，用时 {time.time()-t0:.1f}s")

In [ ]:
##### 宽度扫描：深度固定 3，宽度 4~512，n=40，30000 步 #####
cfg_mem = dict(CFG, n_train=40, iters=30000)
print("检查：", cfg_mem["n_train"], cfg_mem["iters"])      # 应为 40 30000

WIDTHS = [4, 8, 16, 32, 64, 128, 256, 512]
for s in [0, 1, 2]:
    for r in [1.5, 3.0]:
        for w in WIDTHS:
            run_custom(r, 3, w, s, cfg_mem, tag="widthsweep")

In [ ]:
files = glob.glob(f"{OUT_DIR}/widthsweep/*.npz")
df = pd.read_csv(RESULT_CSV); m = df[df.tag == "widthsweep"]
print("npz 数量：", len(files), "| csv 行数：", len(m), "| 重复：", m.duplicated(["radius","width","seed"]).sum())
print(m.groupby(["radius", "width"]).size().unstack())    # 应为 2 行 8 列，全是 3

In [ ]:
a = df[(df.tag == "widthsweep") & (df.width == 64) & (df.radius == 3.0)][["seed", "final_loss"]].sort_values("seed")
b = df[(df.tag == "mainN40") & (df.dist_idx == 3) & (df.cap_idx == 2)][["seed", "final_loss"]].sort_values("seed")
print(a.merge(b, on="seed", suffixes=("_扫描", "_mainN40")))

#### 12.消融

##### 训练样本量消融

In [ ]:
DISTANCES = [1.0, 1.5, 2.0, 3.0, 4.5]

NS = [20, 100, 300, 1000]                  # n=40 直接用 mainN40 的结果
for s in [0, 1, 2]:
    for r in [1.5, 3.0]:
        for n in NS:
            cfg = dict(CFG, n_train=n, iters=30000)
            print("检查：", cfg["n_train"], cfg["iters"])
            run_custom(r, 4, 256, s, cfg, tag=f"ablN{n}")

In [ ]:
##### 核对 #####
for n in [20, 100, 300, 1000]:
    m = df_new = pd.read_csv(RESULT_CSV); m = m[m.tag == f"ablN{n}"]
    print(f"ablN{n}：", len(m), "行，重复", m.duplicated(["radius", "seed"]).sum())   # 每个 n 应为 6 行、重复 0

##### 扩散步数消融

In [ ]:
run_ablation("T", [50, 200], di=2, ci=2, seeds=(0, 1))   # T=100 用 main 里 d2 c2 的现成结果

In [ ]:
##### 核对#####
df = pd.read_csv(RESULT_CSV)
print(df[df.tag.str.startswith("abl_T")].groupby("tag").size())     # abl_T50、abl_T200 各 2 行
print(df[(df.tag=="main") & (df.dist_idx==2) & (df.cap_idx==2) & (df.seed<2)][["seed","final_loss"]])  # T=100 的对照
print(df[df.tag.str.startswith("abl_T")][["tag","seed","final_loss"]])

#### 13.训练集与新样本上的去噪损失差（泛化间隙）

In [ ]:
DISTANCES  = [1.0, 1.5, 2.0, 3.0, 4.5]
CAPACITIES = [(1, 8), (2, 16), (3, 64), (4, 256), (5, 512)]

def denoise_loss(model, x_np, n_rep=100, seed=0):
    """固定随机数，多次加噪取平均，得到稳定的去噪损失。"""
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    d = model.ddpm
    x = torch.tensor(x_np, device=DEVICE)
    model.eval(); tot = 0.0
    with torch.no_grad():
        for _ in range(n_rep):
            t = torch.randint(0, d.T, (len(x),), device=DEVICE, generator=g)
            eps = torch.randn(x.shape, device=DEVICE, generator=g)
            ab = d.abar[t][:, None]
            xt = ab.sqrt() * x + (1 - ab).sqrt() * eps
            tot += ((model(xt, t) - eps) ** 2).mean().item()
    return tot / n_rep

rows = []
for tag, n in [("main", 1000), ("mainN40", 40)]:
    for di in range(5):
        for ci in range(5):
            for s in range(5):
                pt = f"{OUT_DIR}/{tag}/{tag}_d{di}_c{ci}_s{s}.pt"
                if not os.path.exists(pt):
                    continue
                depth, width = CAPACITIES[ci]
                model = build_model(depth, width, CFG)
                model.load_state_dict(torch.load(pt, map_location=DEVICE))
                r = DISTANCES[di]
                x_tr, _, _ = make_gmm(n, 8, r, 0.3, s)          # 与训练时完全相同的训练集
                x_te, _, _ = make_gmm(2000, 8, r, 0.3, 30000 + s)  # 新的独立样本
                tr = denoise_loss(model, x_tr); te = denoise_loss(model, x_te)
                rows.append(dict(tag=tag, dist_idx=di, cap_idx=ci, seed=s,
                                 train_loss=tr, test_loss=te, gap=te - tr))

gap_df = pd.DataFrame(rows)
gap_df.to_csv(f"{OUT_DIR}/gap_{WHO}.csv", index=False)
print(gap_df.groupby("tag").size())
print(gap_df[gap_df.tag == "mainN40"].pivot_table(index="dist_idx", columns="cap_idx", values="gap"))

In [ ]:
gap_df["rel_gap"] = gap_df["gap"] / gap_df["test_loss"]

for tag in ["main", "mainN40"]:
    g = gap_df[gap_df.tag == tag]
    print(f"\n===== {tag}：绝对差距（5 个种子的平均）=====")
    print(g.pivot_table(index="dist_idx", columns="cap_idx", values="gap").round(4))
    print(f"===== {tag}：相对差距 = 差距 ÷ 新样本损失 =====")
    print(g.pivot_table(index="dist_idx", columns="cap_idx", values="rel_gap").round(3))

g = gap_df[gap_df.tag == "mainN40"]
print("\n===== mainN40：训练损失 =====")
print(g.pivot_table(index="dist_idx", columns="cap_idx", values="train_loss").round(3))
print("===== mainN40：新样本损失 =====")
print(g.pivot_table(index="dist_idx", columns="cap_idx", values="test_loss").round(3))
print("===== mainN40：差距的种子间标准差 =====")
print(g.pivot_table(index="dist_idx", columns="cap_idx", values="gap", aggfunc="std").round(4))

In [ ]:
g = gap_df[gap_df.tag == "mainN40"].copy()
g["rel_gap"] = g["gap"] / g["test_loss"]

# 1) (4,256) → (5,512) 的下降，按种子配对
p = g.pivot_table(index=["dist_idx", "seed"], columns="cap_idx", values="gap")
d = (p[4] - p[3]).groupby("dist_idx").agg(["mean", "std", "count"])
d["se"] = d["std"] / np.sqrt(d["count"]); d["z"] = d["mean"] / d["se"]
print("最大模型的变化（负数=下降）：\n", d.round(4))

# 2) 距离 4.5 对距离 1.0 的相对差距，按种子配对（取 (4,256)）
q = g[g.cap_idx == 3].pivot_table(index="seed", columns="dist_idx", values="rel_gap")
dd = q[4] - q[0]
print("\n(4,256)：r=4.5 减 r=1.0 的相对差距，均值", round(dd.mean(), 3),
      "标准误", round(dd.std() / np.sqrt(len(dd)), 3))

In [ ]:
##### 合并检验 #####
from scipy import stats

p = g.pivot_table(index=["dist_idx", "seed"], columns="cap_idx", values="gap")
delta = (p[4] - p[3]).groupby("seed").mean()        # 每个种子：五个距离平均后的变化
t, pv = stats.ttest_1samp(delta, 0)
se = delta.std() / np.sqrt(len(delta))
print("各种子的变化：", delta.round(4).tolist())
print("均值", round(delta.mean(), 4), "| t =", round(t, 2), "| p =", round(pv, 4))
print("95% 置信区间：", round(delta.mean() - stats.t.ppf(.975, 4) * se, 4),
      "到", round(delta.mean() + stats.t.ppf(.975, 4) * se, 4))

In [ ]:
##### 对照 #####
DISTANCES  = [1.0, 1.5, 2.0, 3.0, 4.5]
CAPACITIES = [(1, 8), (2, 16), (3, 64), (4, 256), (5, 512)]
cfg60 = dict(CFG, n_train=40, iters=60000)
print("检查：", cfg60["n_train"], cfg60["iters"])          # 应为 40 60000
for s in [0, 1, 2]:
    for di in [1, 3]:                                       # r=1.5 和 r=3.0
        run_one(di, 4, s, cfg=cfg60, tag="iter60k")

In [ ]:
rows = []
for di in [1, 3]:
    for s in [0, 1, 2]:
        pt = f"{OUT_DIR}/iter60k/iter60k_d{di}_c4_s{s}.pt"
        model = build_model(5, 512, CFG)
        model.load_state_dict(torch.load(pt, map_location=DEVICE))
        r = DISTANCES[di]
        x_tr, _, _ = make_gmm(40, 8, r, 0.3, s)
        x_te, _, _ = make_gmm(2000, 8, r, 0.3, 30000 + s)
        tr = denoise_loss(model, x_tr); te = denoise_loss(model, x_te)
        rows.append(dict(dist_idx=di, seed=s, train_loss=tr, test_loss=te, gap=te - tr, rel_gap=(te - tr) / te))
it60 = pd.DataFrame(rows)
it60.to_csv(f"{OUT_DIR}/gap_iter60k_{WHO}.csv", index=False)

# 与 mainN40 里同种子、同距离的 (5,512)（30000 步）和 (4,256) 对比
ref = gap_df[(gap_df.tag == "mainN40") & (gap_df.dist_idx.isin([1, 3])) & (gap_df.seed < 3)].copy()
ref["rel_gap"] = ref["gap"] / ref["test_loss"]
print("===== 60000 步的 (5,512)：相对差距、训练损失 =====")
print(it60.groupby("dist_idx")[["train_loss", "test_loss", "gap", "rel_gap"]].mean().round(4))
print("===== mainN40 同种子（0–2）的 (4,256) 和 (5,512) =====")
print(ref[ref.cap_idx.isin([3, 4])].groupby(["dist_idx", "cap_idx"])[["train_loss", "test_loss", "gap", "rel_gap"]].mean().round(4))

In [ ]:
##### 对照 #####
DISTANCES  = [1.0, 1.5, 2.0, 3.0, 4.5]
CAPACITIES = [(1, 8), (2, 16), (3, 64), (4, 256), (5, 512)]
cfg60 = dict(CFG, n_train=40, iters=60000)
print("检查：", cfg60["n_train"], cfg60["iters"])          # 应为 40 60000
for s in [0, 1, 2]:
    for di in [1, 3]:
        run_one(di, 3, s, cfg=cfg60, tag="iter60k")         # 容量下标 3 = (4,256)

In [ ]:
rows = []
for ci in [3, 4]:
    for di in [1, 3]:
        for s in [0, 1, 2]:
            pt = f"{OUT_DIR}/iter60k/iter60k_d{di}_c{ci}_s{s}.pt"
            depth, width = CAPACITIES[ci]
            model = build_model(depth, width, CFG)
            model.load_state_dict(torch.load(pt, map_location=DEVICE))
            r = DISTANCES[di]
            x_tr, _, _ = make_gmm(40, 8, r, 0.3, s)
            x_te, _, _ = make_gmm(2000, 8, r, 0.3, 30000 + s)
            tr = denoise_loss(model, x_tr); te = denoise_loss(model, x_te)
            rows.append(dict(cap_idx=ci, dist_idx=di, seed=s, train_loss=tr,
                             test_loss=te, gap=te - tr, rel_gap=(te - tr) / te))
it60 = pd.DataFrame(rows)
it60.to_csv(f"{OUT_DIR}/gap_iter60k_{WHO}.csv", index=False)
print(it60.groupby(["dist_idx", "cap_idx"])[["train_loss", "test_loss", "gap", "rel_gap"]].mean().round(4))

In [ ]:
p = it60.pivot_table(index=["dist_idx", "seed"], columns="cap_idx", values="gap")
diff = (p[4] - p[3])
print(diff.round(4)); print("下降的配对数：", (diff < 0).sum(), "/", len(diff))

In [ ]:
gap_df["rel_gap"] = gap_df["gap"] / gap_df["test_loss"]
labels = [f"({d},{w})" for d, w in CAPACITIES]
fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
for k, (tag, title) in enumerate([("main", "n = 1000"), ("mainN40", "n = 40")]):
    g = gap_df[gap_df.tag == tag]
    for di in range(5):
        m = g[g.dist_idx == di].groupby("cap_idx")["rel_gap"].agg(["mean", "std"])
        ax[k].errorbar(range(5), m["mean"], yerr=m["std"] / np.sqrt(5), marker="o",
                       capsize=2, label=f"r = {DISTANCES[di]}")
    ax[k].set_xticks(range(5)); ax[k].set_xticklabels(labels, fontsize=8)
    ax[k].set_title(title); ax[k].set_xlabel("capacity (depth, width)")
ax[0].set_ylabel("relative loss gap (test - train) / test"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_gap.png", dpi=200); plt.show()

#### 14.深度扫瞄

In [ ]:
DISTANCES  = [1.0, 1.5, 2.0, 3.0, 4.5]
CAPACITIES = [(1, 8), (2, 16), (3, 64), (4, 256), (5, 512)]

def run_custom_save(radius, depth, width, seed, cfg, tag):
    name = f"{tag}_r{radius}_d{depth}_w{width}_s{seed}"
    folder = os.path.join(OUT_DIR, tag); os.makedirs(folder, exist_ok=True)
    npz = os.path.join(folder, name + ".npz"); pt = os.path.join(folder, name + ".pt")
    ckpt = os.path.join(folder, name + ".ckpt")
    if os.path.exists(npz):
        print("跳过（已完成）：", name); return
    t0 = time.time()
    model, losses, x_train = train(depth, width, radius, seed, cfg, ckpt)
    final, traj = sample(model, cfg["n_sample"], True, seed=seed, n_traj=cfg["n_traj"])
    torch.save(model.state_dict(), pt)
    np.savez_compressed(npz, final=final, traj=traj, train=x_train,
                        losses=np.array(losses), means_radius=radius)
    append_row(dict(tag=tag, dist_idx=DISTANCES.index(radius), cap_idx=-1, radius=radius,
                    depth=depth, width=width, params=n_params(model), seed=seed,
                    n_train=cfg["n_train"], T=cfg["T"], K=cfg["K"],
                    final_loss=round(float(np.mean(losses[-200:])), 5),
                    seconds=round(time.time() - t0, 1), npz_path=npz))
    if os.path.exists(ckpt): os.remove(ckpt)
    print(f"完成 {name}：参数量 {n_params(model)}，用时 {time.time()-t0:.1f}s")

cfg_mem = dict(CFG, n_train=40, iters=30000)
print("检查：", cfg_mem["n_train"], cfg_mem["iters"])      # 应为 40 30000
for s in [0, 1, 2]:
    for r in [1.5, 3.0]:
        for depth in [1, 2, 3, 4, 5, 6]:
            run_custom_save(r, depth, 256, s, cfg_mem, tag="depthsweep")

In [ ]:
files = glob.glob(f"{OUT_DIR}/depthsweep/*.npz")
df = pd.read_csv(RESULT_CSV); m = df[df.tag == "depthsweep"]
print("npz：", len(files), "| csv：", len(m), "| 重复：", m.duplicated(["radius","depth","seed"]).sum())   # 36 | 36 | 0

rows = []
for r in [1.5, 3.0]:
    for depth in range(1, 7):
        for s in range(3):
            pt = f"{OUT_DIR}/depthsweep/depthsweep_r{r}_d{depth}_w256_s{s}.pt"
            model = build_model(depth, 256, CFG)
            model.load_state_dict(torch.load(pt, map_location=DEVICE))
            x_tr, _, _ = make_gmm(40, 8, r, 0.3, s)
            x_te, _, _ = make_gmm(2000, 8, r, 0.3, 30000 + s)
            tr = denoise_loss(model, x_tr); te = denoise_loss(model, x_te)
            rows.append(dict(r=r, depth=depth, seed=s, train_loss=tr, test_loss=te,
                             gap=te - tr, rel_gap=(te - tr) / te))
dsw = pd.DataFrame(rows); dsw.to_csv(f"{OUT_DIR}/gap_depthsweep_{WHO}.csv", index=False)
print(dsw.pivot_table(index="depth", columns="r", values="rel_gap").round(3))

In [ ]:
##### 检查 #####
a = dsw[dsw.depth == 5].set_index(["r", "seed"])["rel_gap"]
b = gap_df[(gap_df.tag == "mainN40") & (gap_df.cap_idx == 4) & (gap_df.dist_idx.isin([1, 3])) & (gap_df.seed < 3)].copy()
b["rel_gap"] = b["gap"] / b["test_loss"]
b["r"] = b["dist_idx"].map({1: 1.5, 3: 3.0})
b = b.set_index(["r", "seed"])["rel_gap"]
diff = (b - a).dropna()
print(diff.round(3)); print("下降的配对数：", (diff < 0).sum(), "/", len(diff))

In [ ]:
##### 宽度固定 512，深度从 3 加到 6 #####
print("run_custom_save" in globals())

DISTANCES  = [1.0, 1.5, 2.0, 3.0, 4.5]
cfg_mem = dict(CFG, n_train=40, iters=30000)
print("检查：", cfg_mem["n_train"], cfg_mem["iters"])      # 应为 40 30000
for s in [0, 1, 2]:
    for r in [1.5, 3.0]:
        for depth in [3, 4, 6]:
            run_custom_save(r, depth, 512, s, cfg_mem, tag="depthsweep512")

In [ ]:
files = glob.glob(f"{OUT_DIR}/depthsweep512/*.npz")
df = pd.read_csv(RESULT_CSV); m = df[df.tag == "depthsweep512"]
print("npz：", len(files), "| csv：", len(m), "| 重复：", m.duplicated(["radius","depth","seed"]).sum())  # 18 | 18 | 0

rows = []
for r in [1.5, 3.0]:
    for depth in [3, 4, 6]:
        for s in range(3):
            pt = f"{OUT_DIR}/depthsweep512/depthsweep512_r{r}_d{depth}_w512_s{s}.pt"
            model = build_model(depth, 512, CFG)
            model.load_state_dict(torch.load(pt, map_location=DEVICE))
            x_tr, _, _ = make_gmm(40, 8, r, 0.3, s)
            x_te, _, _ = make_gmm(2000, 8, r, 0.3, 30000 + s)
            tr = denoise_loss(model, x_tr); te = denoise_loss(model, x_te)
            rows.append(dict(r=r, depth=depth, seed=s, rel_gap=(te - tr) / te))
d512 = pd.DataFrame(rows)

# 深度 5 取 mainN40 的 (5,512)
g5 = gap_df[(gap_df.tag == "mainN40") & (gap_df.cap_idx == 4) & (gap_df.dist_idx.isin([1, 3])) & (gap_df.seed < 3)].copy()
g5["rel_gap"] = g5["gap"] / g5["test_loss"]; g5["r"] = g5["dist_idx"].map({1: 1.5, 3: 3.0}); g5["depth"] = 5
d512 = pd.concat([d512, g5[["r", "depth", "seed", "rel_gap"]]])
d512.to_csv(f"{OUT_DIR}/gap_depthsweep512_{WHO}.csv", index=False)
print(d512.pivot_table(index="depth", columns="r", values="rel_gap").round(3))

In [ ]:
import matplotlib.pyplot as plt, numpy as np

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8), sharey=False)
for k, r in enumerate([1.5, 3.0]):
    a = dsw[dsw.r == r].groupby("depth")["rel_gap"].agg(["mean", "std"])
    b = d512[d512.r == r].groupby("depth")["rel_gap"].agg(["mean", "std"])
    ax[k].errorbar(a.index, a["mean"], yerr=a["std"] / np.sqrt(3), marker="o", capsize=2, label="width 256")
    ax[k].errorbar(b.index, b["mean"], yerr=b["std"] / np.sqrt(3), marker="s", capsize=2, label="width 512")
    ax[k].set_title(f"r = {r}"); ax[k].set_xlabel("depth")
ax[0].set_ylabel("relative loss gap (test - train) / test"); ax[0].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_depth_width.png", dpi=200); plt.show()

In [ ]:
def curve(tag, di, ci, s):
    d = np.load(f"{OUT_DIR}/{tag}/{tag}_d{di}_c{ci}_s{s}.npz")["losses"]
    return d

print("训练损失（200 步平均），mainN40，种子 0–2 平均")
print("距离 | 步数 |  (4,256) | (5,512)")
for di, r in [(1, 1.5), (3, 3.0)]:
    for step in [2000, 5000, 10000, 20000, 30000]:
        v = {}
        for ci in [3, 4]:
            v[ci] = np.mean([curve("mainN40", di, ci, s)[step - 200:step].mean() for s in range(3)])
        print(f"r={r} | {step:>5} | {v[3]:.4f} | {v[4]:.4f}")